# 🔐 FaceVault — Colab Demo

**Fast face recognition with anti-spoofing and vector database.**

This notebook walks through the full FaceVault workflow:
1. Install dependencies
2. Clone the repo
3. Register identities from a dataset folder
4. Identify unknown faces (both modes)
5. Generate stamped & side-by-side images

---

## 1. Install & Setup

In [ ]:
# Clone the repository
!git clone https://github.com/YOUR_USERNAME/face_vault.git
%cd face_vault

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
# GPU acceleration (optional — falls back to CPU)
!pip install -q onnxruntime-gpu

In [ ]:
# Verify installation
import face_vault
print(f"FaceVault v{face_vault.__version__} loaded.")

## 2. Download Sample Faces

We'll use a few sample face images for the demo.
You can replace these with your own images.

In [ ]:
import os
import urllib.request
from pathlib import Path

# Create dataset structure
dataset_dir = Path("dataset")

# Sample face URLs (replace with your own)
SAMPLE_FACES = {
    "USR-001": {
        "name": "Person A",
        "urls": [
            "https://thispersondoesnotexist.com",
        ]
    },
    "USR-002": {
        "name": "Person B",
        "urls": [
            "https://thispersondoesnotexist.com",
        ]
    },
}

print("\n\u26a0\ufe0f  NOTE: thispersondoesnotexist.com generates random faces.")
print("   For a real test, upload your own images to dataset/<user_code>/")
print("   using the Colab file browser (left sidebar).\n")

# Create folders
for code in SAMPLE_FACES:
    (dataset_dir / code).mkdir(parents=True, exist_ok=True)

print("\u2705 Dataset folders created:")
for d in sorted(dataset_dir.iterdir()):
    print(f"   {d}/")

### ⬆️ Upload Your Images

Use the **file browser** (left sidebar) to upload face images into:
- `dataset/USR-001/` — all photos of Person A
- `dataset/USR-002/` — all photos of Person B

Or use the upload widget below:

In [ ]:
from google.colab import files

# Upload images for Person A
print("Upload images for USR-001 (Person A):")
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f"dataset/USR-001/{name}", "wb") as f:
        f.write(data)
    print(f"  \u2713 Saved: dataset/USR-001/{name}")

In [ ]:
# Upload images for Person B
print("Upload images for USR-002 (Person B):")
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f"dataset/USR-002/{name}", "wb") as f:
        f.write(data)
    print(f"  \u2713 Saved: dataset/USR-002/{name}")

## 3. Initialize FaceVault

In [ ]:
from face_vault import FaceVault, DetectMode
import cv2
from IPython.display import display, Image as IPImage
import numpy as np

# Create the vault
vault = FaceVault(
    db_path="demo_faces.db",
    dataset_dir="dataset",
    ctx_id=0,              # 0 = GPU, -1 = CPU
    det_size=(640, 640),
    match_threshold=0.4,
    anti_spoof=True,
    spoof_block=False,     # Don't block — just warn (for demo)
)
print("\u2705 FaceVault ready!")

## 4. Register Identities

Register faces from the dataset folders.
Each folder = one person, all images inside belong to them.

In [ ]:
# Register Person A — from folder
result_a = vault.register(
    full_name="Person A",
    user_code="USR-001",
    reference_image="dataset/USR-001/",
)
print(f"Person A: {result_a.message}")

# Register Person B — from folder
result_b = vault.register(
    full_name="Person B",
    user_code="USR-002",
    reference_image="dataset/USR-002/",
)
print(f"Person B: {result_b.message}")

In [ ]:
# Check registered identities
for identity in vault.list_identities():
    print(f"  [{identity.user_code}] {identity.name} — "
          f"{identity.num_vectors} vector(s), ref: {identity.reference_image_path}")

print(f"\nStats: {vault.stats()}")

## 5. Identify — DETECT_WITH_IDENTITY

Upload an unknown face to identify.
The system searches the **entire** database.

In [ ]:
# Upload an unknown face
print("Upload an image to identify:")
uploaded = files.upload()
unknown_path = list(uploaded.keys())[0]
print(f"\u2713 Using: {unknown_path}")

In [ ]:
# Identify with overlay + reference side-by-side
result = vault.identify(
    unknown_path,
    mode=DetectMode.DETECT_WITH_IDENTITY,
    image_overlay=True,
    reference_image=True,
)

print(f"Matched:    {result.matched}")
print(f"Name:       {result.name}")
print(f"User Code:  {result.user_code}")
print(f"Similarity: {result.similarity:.4f}")
print(f"Time:       {result.elapsed_ms:.1f} ms")

if result.spoof_result:
    sr = result.spoof_result
    print(f"Spoof:      {sr.verdict.value} (score={sr.score:.3f})")

# Display the annotated image
if result.image is not None:
    _, buf = cv2.imencode('.jpg', result.image)
    display(IPImage(data=buf.tobytes()))
    cv2.imwrite("result_with_identity.jpg", result.image)
    print("\u2713 Saved: result_with_identity.jpg")

## 6. Identify — DETECT_WITHOUT_IDENTITY

Check if the unknown face matches a **specific** user.
This is the fast path — only checks that user's vectors.

In [ ]:
# Fast 1:1 check — is this USR-001?
result_check = vault.identify(
    unknown_path,
    user_code="USR-001",
    mode=DetectMode.DETECT_WITHOUT_IDENTITY,
    image_overlay=True,
)

print(f"Is this USR-001? {result_check.matched}")
print(f"Similarity:      {result_check.similarity:.4f}")
print(f"Time:            {result_check.elapsed_ms:.1f} ms")

if result_check.image is not None:
    _, buf = cv2.imencode('.jpg', result_check.image)
    display(IPImage(data=buf.tobytes()))
    cv2.imwrite("result_without_identity.jpg", result_check.image)
    print("\u2713 Saved: result_without_identity.jpg")

## 7. Overlay Only (No Reference)

Get the stamped image without the side-by-side reference.

In [ ]:
# Stamped overlay only — no side-by-side
result_overlay = vault.identify(
    unknown_path,
    image_overlay=True,
    reference_image=False,  # no side-by-side
)

if result_overlay.image is not None:
    _, buf = cv2.imencode('.jpg', result_overlay.image)
    display(IPImage(data=buf.tobytes()))
    print("\u2713 Overlay only (no reference image)")

## 8. 1:1 Verification

Compare two specific images directly.

In [ ]:
# Upload two images to compare
print("Upload Image A:")
up_a = files.upload()
path_a = list(up_a.keys())[0]

print("Upload Image B:")
up_b = files.upload()
path_b = list(up_b.keys())[0]

vr = vault.verify(path_a, path_b)
print(f"\nSame person:  {vr.is_same_person}")
print(f"Similarity:   {vr.similarity:.4f}")
print(f"Distance:     {vr.distance:.4f}")
print(f"Time:         {vr.elapsed_ms:.1f} ms")
if vr.spoof_result:
    print(f"Spoof:        {vr.spoof_result.verdict.value}")

## 9. Database Stats & Cleanup

In [ ]:
# Show database stats
print("Database Statistics:")
for k, v in vault.stats().items():
    print(f"  {k}: {v}")

print("\nRegistered Identities:")
for ident in vault.list_identities():
    print(f"  [{ident.user_code}] {ident.name} — {ident.num_vectors} vectors")

In [ ]:
# Cleanup
vault.close()
print("\u2705 FaceVault closed.")

---

## API Quick Reference

```python
from face_vault import FaceVault, DetectMode

vault = FaceVault("faces.db", dataset_dir="dataset")

# Register (folder of images)
vault.register(
    full_name="Afiya Kelifa Ahimed",
    user_code="OGH-00238",
    reference_image="dataset/OGH-00238/",
)

# Register (auto-discover from dataset/<user_code>/)
vault.register(full_name="John Doe", user_code="OGH-00100")

# Identify — who is this?
r = vault.identify("unknown.jpg")
print(r.matched, r.name, r.user_code)

# Identify — with annotated image
r = vault.identify("unknown.jpg", image_overlay=True, reference_image=True)
cv2.imwrite("result.jpg", r.image)

# Identify — is this user X?
r = vault.identify("unknown.jpg", user_code="OGH-00238",
                   mode=DetectMode.DETECT_WITHOUT_IDENTITY)
print(r.matched)

# Verify — same person?
r = vault.verify("a.jpg", "b.jpg")
print(r.is_same_person, r.similarity)
```